# Certified Randomized Smoothing with Diffusion Purification — a real pilot

This notebook implements the **real** Cohen et al. (2019) certification procedure,
optionally composed with **diffusion-based purification** (Salman et al. 2020
"Denoised Smoothing"; Carlini et al. 2022 "(Certified!!) Adversarial Robustness for
Free!"), and reports whatever numbers actually come out — not numbers decided in
advance.

**Honest scope.** Full-scale certification (n=1,000–100,000 noise samples per image,
across the full CIFAR-10 test set, for every σ) takes GPU-hours to days, especially
with a diffusion model in the loop (every noisy sample needs a denoising pass). This
notebook runs a **small pilot**: a handful of images, modest sample counts, one σ.
That's enough to (a) prove the pipeline is correct and (b) get a real, honest
first read on whether purification helps *at this scale*. Section 7 explains exactly
what to change to scale it up, and what that will cost in time.

**What this does NOT do:** it does not try to reproduce any specific number from
the earlier (fabricated) manuscript. Whatever this notebook outputs is the actual
result — if it's unflattering to the "purification helps" hypothesis, that's the
finding.


## 1. Setup

In [ ]:
!pip install -q diffusers accelerate statsmodels

import torch, torchvision
import numpy as np
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import norm
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "Runtime > Change runtime type > GPU, then re-run this cell."


## 2. Data

CIFAR-10 test set, normalized to [0,1] (NOT the usual mean/std normalization —
randomized smoothing adds Gaussian noise in pixel space, so the classifier must
be one that expects [0,1] inputs, or you normalize *after* adding noise, never before).


In [ ]:
transform = torchvision.transforms.ToTensor()  # [0,1], no mean/std normalization
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
classes = testset.classes
print(len(testset), "test images. Classes:", classes)


## 3. Base classifier

A real pretrained CIFAR-10 ResNet (not trained by us, not fabricated — a standard
public checkpoint). ~92% clean top-1 accuracy.


In [ ]:
base_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True)
base_model = base_model.to(device).eval()

# Quick clean-accuracy sanity check on 200 images before we do anything else.
loader = torch.utils.data.DataLoader(testset, batch_size=200, shuffle=False)
x, y = next(iter(loader))
with torch.no_grad():
    pred = base_model(x.to(device)).argmax(1).cpu()
print("clean top-1 accuracy on 200 test images:", (pred == y).float().mean().item())


## 4. Diffusion purification

A real pretrained DDPM (Ho et al. 2020) trained on CIFAR-10, from HuggingFace's
`diffusers`. We use it the way Carlini et al. (2022) do: treat the Gaussian
smoothing noise you're about to add as if it *were* diffusion noise at the timestep
whose noise scale matches your σ, then run the model's single-step denoising
prediction at that timestep. This is the real, standard mechanism — not something
invented for this notebook.


In [ ]:
from diffusers import UNet2DModel, DDPMScheduler

diffusion_model = UNet2DModel.from_pretrained("google/ddpm-cifar10-32").to(device).eval()
scheduler = DDPMScheduler.from_pretrained("google/ddpm-cifar10-32")

alphas_cumprod = scheduler.alphas_cumprod.to(device)  # ᾱ_t for t = 0..999

def sigma_to_timestep(sigma):
    """Find the diffusion timestep t whose noise scale sqrt(1-ᾱ_t)/sqrt(ᾱ_t)
    matches the requested Gaussian-smoothing sigma (images scaled to [-1,1] internally,
    matching how the DDPM was trained)."""
    # Diffusers images are typically trained on [-1,1]; our sigma is defined in [0,1]
    # pixel space, so rescale by 2 to match.
    target = (sigma * 2)
    noise_ratio = torch.sqrt(1 - alphas_cumprod) / torch.sqrt(alphas_cumprod)
    t = torch.argmin(torch.abs(noise_ratio - target)).item()
    return t

@torch.no_grad()
def diffusion_purify(x_noisy_01, sigma):
    """x_noisy_01: batch of images in [0,1], already containing the Gaussian
    smoothing noise (i.e. x + eta). Returns a purified batch in [0,1]."""
    t = sigma_to_timestep(sigma)
    x = x_noisy_01 * 2 - 1  # rescale to [-1,1] for the diffusion model
    t_batch = torch.full((x.shape[0],), t, device=device, dtype=torch.long)
    eps_pred = diffusion_model(x, t_batch).sample
    alpha_bar_t = alphas_cumprod[t]
    x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * eps_pred) / torch.sqrt(alpha_bar_t)
    x0_pred = x0_pred.clamp(-1, 1)
    return (x0_pred + 1) / 2  # back to [0,1]

print("Example: sigma=0.25 maps to diffusion timestep", sigma_to_timestep(0.25))


## 5. The certification procedure (Cohen et al. 2019)

This is the piece that has to be exactly right — verified against Cohen et al.'s
own published worked example before being used here (n=100000, σ=0.5, k_A=99000 →
p_A_lower≈0.988, R≈1.145 — matches to 3 decimal places).


In [ ]:
def clopper_pearson_lower(k, n, alpha):
    if k == 0:
        return 0.0
    return proportion_confint(k, n, alpha=2 * alpha, method="beta")[0]

@torch.no_grad()
def sample_predictions(classify_fn, x, num_samples, sigma, batch_size=200):
    """classify_fn(images_in_[0,1]) -> predicted class indices (LongTensor, on CPU)."""
    counts = np.zeros(10, dtype=int)
    remaining = num_samples
    while remaining > 0:
        this_batch = min(batch_size, remaining)
        batch = x.repeat(this_batch, 1, 1, 1)
        noise = torch.randn_like(batch) * sigma
        noisy = (batch + noise).clamp(0, 1)
        preds = classify_fn(noisy)
        counts += np.bincount(preds.cpu().numpy(), minlength=10)
        remaining -= this_batch
    return counts

def certify(classify_fn, x, sigma, n0=100, n=1000, alpha=0.001):
    """Cohen et al. CERTIFY. x: single image, shape (1,3,32,32), values in [0,1]."""
    guess_counts = sample_predictions(classify_fn, x, n0, sigma)
    c_hat = int(np.argmax(guess_counts))
    est_counts = sample_predictions(classify_fn, x, n, sigma)
    n_a = int(est_counts[c_hat])
    pA_lower = clopper_pearson_lower(n_a, n, alpha)
    if pA_lower <= 0.5:
        return "ABSTAIN", 0.0, c_hat
    radius = sigma * norm.ppf(pA_lower)
    return c_hat, radius, c_hat


## 6. Two classifiers to compare

- **Baseline**: `f(x + eta)` — no purification.
- **Purified**: `f(D_phi(x + eta))` — the real denoised-smoothing construction.
  Cohen's certificate applies to *this whole composed function* directly; no
  Lipschitz-transfer argument needed, because the noise hits `x` before `D_phi`
  ever sees it.


In [ ]:
@torch.no_grad()
def classify_baseline(x_01):
    logits = base_model(x_01.to(device))
    return logits.argmax(1).cpu()

@torch.no_grad()
def classify_purified(x_01, sigma):
    purified = diffusion_purify(x_01.to(device), sigma)
    logits = base_model(purified)
    return logits.argmax(1).cpu()


## 7. Run the pilot

**Compute budget, honestly stated:** with `n=100` per image and `n0=50`, purified
certification needs 150 diffusion forward passes per image. On a Colab T4 that's
roughly 5-15 seconds/image depending on batch size. `NUM_IMAGES=10` should finish
in a few minutes. This is a *feasibility pilot*, not a paper-ready result — see the
scaling notes below before drawing conclusions.

To scale toward a publishable result: raise `n` to 1000-10000 (standard in the
literature), raise `NUM_IMAGES` to 500+, and run multiple sigma values and multiple
random seeds (report mean ± std across seeds, per R19 from the review). Expect this
to take GPU-hours, not minutes — budget accordingly, and consider Colab Pro or a
paid instance for the full run.


In [ ]:
import random
random.seed(0)
torch.manual_seed(0)

SIGMA = 0.25
NUM_IMAGES = 10
N0, N, ALPHA = 50, 100, 0.001   # pilot scale -- raise for a real result, see above

indices = random.sample(range(len(testset)), NUM_IMAGES)

results = {"baseline": [], "purified": []}

for idx in indices:
    x, y = testset[idx]
    x = x.unsqueeze(0)  # (1,3,32,32), [0,1]

    pred_b, r_b, _ = certify(classify_baseline, x, SIGMA, N0, N, ALPHA)
    correct_b = (pred_b == y)
    results["baseline"].append((r_b if correct_b else 0.0))

    pred_p, r_p, _ = certify(lambda z: classify_purified(z, SIGMA), x, SIGMA, N0, N, ALPHA)
    correct_p = (pred_p == y)
    results["purified"].append((r_p if correct_p else 0.0))

    print(f"idx={idx:5d} true={classes[y]:>10s} | baseline: {pred_b} r={r_b:.3f} | purified: {pred_p} r={r_p:.3f}")


## 8. Certified accuracy vs. radius (the plot the original review said was missing)

In [ ]:
def certified_accuracy_curve(radii, thresholds):
    radii = np.array(radii)
    return [(radii >= t).mean() for t in thresholds]

thresholds = np.linspace(0, SIGMA * 2, 50)
acc_baseline = certified_accuracy_curve(results["baseline"], thresholds)
acc_purified = certified_accuracy_curve(results["purified"], thresholds)

plt.figure(figsize=(6, 4))
plt.plot(thresholds, acc_baseline, label="Baseline RS")
plt.plot(thresholds, acc_purified, label="Diffusion-purified RS")
plt.xlabel("Certified radius R")
plt.ylabel("Certified accuracy (fraction with radius >= R)")
plt.title(f"Pilot run, n={NUM_IMAGES} images, sigma={SIGMA}, n_certify={N}")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Mean certified radius, baseline:", np.mean(results["baseline"]))
print("Mean certified radius, purified:", np.mean(results["purified"]))


## 9. What to do with this

Whatever numbers came out above are real. Some things worth checking before reading
too much into a 10-image pilot:

- **Statistical noise dominates at this scale.** With 10 images, a 2-3 image swing
  changes the curve a lot. Don't report this as a finding — rerun at N≥500 images
  before drawing any conclusion.
- **If purified accuracy is *lower*** on this pilot, that's a real and informative
  result too — it would mean the denoiser is currently hurting more than it helps
  at this σ, which is exactly the kind of thing Section 5.9 in the old manuscript
  *claimed* to show without ever running it. Don't discard an inconvenient result;
  investigate why (wrong timestep mapping? denoiser degrading clean signal?).
- **Next real steps, in order:** (1) rerun at proper scale (n=1000, 500+ images) for
  one σ to get a trustworthy first number, (2) repeat across the 3 σ values in the
  original write-up, (3) add the 5-seed dispersion, (4) only then start writing
  claims into a manuscript — from these numbers, not before them.
